# **04_16_ring.ipynb**

**Programmers:**
* Albonia, Jade Lorenz M.
* Caspe, Mark Vincent G.
* Rivera, Rei Djemf M.
* Velante, Kamilah Kaye M.
* Villegas, Jedidiah S.


**Date Written:** July 2025

**Date Revised:** December 2025

---

### **System Context**
This notebook serves as the **Experimental Execution Harness** hosted on Google Colab. It functions as the orchestration layer that integrates the A-EYE source code (`src/`), the clinical dataset, and cloud-based GPU resources (NVIDIA T4). Instead of relying on manual CLI commands, this environment automates the end-to-end training pipeline to ensure strict reproducibility of the study's results.

### **Purpose**
To establish a **controlled training environment** for the **Baseline / A-EYE** model. This notebook automates:
1.  **Dependency Provisioning:** Installing specific library versions defined in `requirements.txt`.
2.  **Dynamic Logging:** Injecting real-time monitoring hooks into the training script.
3.  **Execution:** Invoking the `train.py` routine with the precise hyperparameters used in the experiments.

---

### **Technical Architecture (Data Structures & Algorithms)**

**1. Infrastructure Management**
* **Ephemeral Workspace (`/content`):** Utilizes the VM's high-speed local disk for data staging and I/O-intensive training operations, avoiding the latency of direct network storage access.
* **Persistent Archival:** Automatically compresses and migrates training artifacts (logs, weights, confusion matrices) to Google Drive to prevent data loss upon runtime disconnection.

**2. Workflow Automation Algorithms**
* **Runtime Code Injection:** Implements a "Hot-Patching" strategy (Cell 4) that dynamically modifies the `scripts/train.py` source code at runtime. This injects custom logging logic to capture training speed (`it/s`) and loss metrics directly in the notebook output stream.
* **Argument Parsing Wrapper:** The `run_training()` function programmatically constructs the `argparse.Namespace` object, allowing notebook variables to interface seamlessly with the command-line logic of the source scripts.

**3. Control Flow**
* **Initialization:** Repository cloning $\rightarrow$ Dependency installation $\rightarrow$ Data staging.
* **Execution:** The notebook encapsulates the training loop within a try-catch block to ensure that errors are logged gracefully without crashing the runtime environment.

---

## Step 1: Mount Google Drive & Unzip Project

In [ ]:
import os
import sys
import warnings

# Suppress all warnings
warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'

REPO_URL = "https://github.com/its-levi0sa/a-eye-cataract-maturity-classification-tool.git"
PROJECT_DIR = "A-EYE"

# --- Clone or Pull Latest Code ---
if os.path.exists(PROJECT_DIR):
    print("Repository already exists. Pulling latest changes...")
    %cd {PROJECT_DIR}
    !git pull
else:
    print("Cloning repository...")
    !git clone {REPO_URL} {PROJECT_DIR}
    %cd {PROJECT_DIR}

# --- Configure Paths and Install Dependencies ---
if os.path.abspath('.') not in sys.path:
    sys.path.insert(0, os.path.abspath('.'))

!pip install -q -r requirements.txt

print("\n✅ Environment setup complete! Ready to train.")

## Step 2: Define Data Path & Training Helper Function

**IMPORTANT:** Update the `DATA_PATH` variable to point to your training dataset folder. This folder should contain the `mature` and `immature` subdirectories.

In [ ]:
import logging
import os
import sys
import argparse
from scripts.train import main as train_main

# --- DATA PATH ---
DATA_PATH = "data/train"

def run_training(model_type, num_rings=None, **kwargs):
    """
    A helper function that runs the training process directly in the notebook
    for perfect logging and progress bar integration.
    """
    # --- 1. Set up the custom logger ---
    log_name = f"training_log_{model_type}" + (f"_{num_rings}_rings" if num_rings else "")
    log_file = f"results/{log_name}.txt"

    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)

    os.makedirs('results', exist_ok=True)
    os.makedirs('saved_models', exist_ok=True)

    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S',
        handlers=[
            logging.FileHandler(log_file, mode='w'),
            logging.StreamHandler(sys.stdout)
        ]
    )

    # --- 2. Create an argument parser object to pass settings to the function ---
    parser = argparse.ArgumentParser()
    args = parser.parse_args(args=[])

    # Populate the args with our settings
    args.model_type = model_type
    args.data_dir = DATA_PATH
    args.num_rings = num_rings

    # Add any extra kwargs like epochs or n_splits, using script defaults if not provided
    args.epochs = kwargs.get('epochs', 150)
    args.n_splits = kwargs.get('n_splits', 5)
    args.batch_size = kwargs.get('batch_size', 16)
    args.learning_rate = kwargs.get('learning_rate', 2e-4)
    args.weight_decay = kwargs.get('weight_decay', 1e-2)
    args.patience = kwargs.get('patience', 20)

    # Add missing args
    args.dims = kwargs.get('dims', [32, 64, 128, 160])
    args.embed_dim = kwargs.get('embed_dim', 256)


    # Proper save dir assignment inside run_training
    if model_type == "baseline":
        args.save_dir = os.path.join("saved_models", "baseline")
    elif model_type == "aeye" and num_rings in [4, 8, 16]:
        args.save_dir = os.path.join("saved_models", f"aeye_{num_rings}_ring")
    else:
        args.save_dir = "saved_models"  # fallback

    os.makedirs(args.save_dir, exist_ok=True)


    logging.info(f"--- Starting Training: {model_type.upper()}" + (f" {num_rings} RINGS" if num_rings else "") + " ---")

    # --- 3. Run the imported training function directly ---
    try:
        train_main(args)
    except Exception as e:
        logging.error(f"An error occurred during training: {e}", exc_info=True)

    logging.info(f"--- Finished Training: {model_type.upper()}" + (f" {num_rings} RINGS" if num_rings else "") + " ---")

print("✅ Training helper function is ready.")

In [ ]:
# Inject logging into train.py if not already there
patch_code = """
    # ✅ log training summary
    final_loss = loss.item()
    avg_speed = train_loop.format_dict.get("rate", 0)
    logging.info(
        f"Epoch {epoch+1} - Train Summary | Speed: {avg_speed:.2f} it/s, Loss: {final_loss:.4f}"
    )
"""

with open("scripts/train.py", "r") as f:
    code = f.read()

if "Train Summary | Speed" not in code:
    code = code.replace(
        "train_loop.set_postfix(loss=loss.item())",
        "train_loop.set_postfix(loss=loss.item())\n" + patch_code
    )
    with open("scripts/train.py", "w") as f:
        f.write(code)
    print("✅ train.py patched to log speed and loss.")
else:
    print("⚡ train.py already patched.")

## Step 3: Run Full Training Sessions

Run the desired cell below to start the **full training process (100 epochs, 5 folds)** for a specific model. This will take a significant amount of time. The output will be displayed in the cell and saved to a dedicated log file in the `results/` directory.

### ▶️ **Train A-EYE Model (16 Rings)**

In [ ]:
run_training(model_type='aeye', num_rings=16, epochs=100, learning_rate=1e-4, patience=20)

## Step 4: Download Final Trained Assets

After all training sessions are complete, this final cell will zip up the `results` and `saved_models` folders. You can then download this single zip file from the Colab file browser on the left. This file will contain all your definitive logs and `.pth` files.

In [ ]:
!zip -r /content/final_assets.zip /content/A-EYE/results /content/A-EYE/saved_models

print("✅ All results and models have been zipped into '/content/final_assets.zip'.")
print("Download it from the file browser on the left before closing this session.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Copy the zip file into MyDrive
!cp /content/final_assets.zip /content/drive/MyDrive/